In [0]:
EVENTHUB_NAMESPACE = "cryptoeventhub.servicebus.windows.net"
EVENTHUB_NAME = "crypto-stream"

BOOTSTRAP_SERVER = f"{EVENTHUB_NAMESPACE}:9093"

ADLS_BASE = "abfss://cryptoblob@cryptoanalysisadls.dfs.core.windows.net"

In [0]:
conn_str = dbutils.secrets.get(scope="crypto-scope", key="eventhub-conn")

SASL_JAAS = (
    "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
    f'username="$ConnectionString" password="{conn_str}";'
)

In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import *

HUB_NAME   = "crypto-stream"

CHECKPOINT = f"{ADLS_BASE}/checkpoint/"
OUTPUT     = f"{ADLS_BASE}/bronze/"

schema = StructType([
    StructField("id", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("name", StringType(), True),
    StructField("image", StringType(), True),
    StructField("current_price", DoubleType(), True),
    StructField("market_cap", DoubleType(), True),
    StructField("market_cap_rank", IntegerType(), True),
    StructField("fully_diluted_valuation", DoubleType(), True),
    StructField("total_volume", DoubleType(), True),
    StructField("high_24h", DoubleType(), True),
    StructField("low_24h", DoubleType(), True),
    StructField("price_change_24h", DoubleType(), True),
    StructField("price_change_percentage_24h", DoubleType(), True),
    StructField("market_cap_change_24h", DoubleType(), True),
    StructField("market_cap_change_percentage_24h", DoubleType(), True),
    StructField("circulating_supply", DoubleType(), True),
    StructField("total_supply", DoubleType(), True),
    StructField("max_supply", DoubleType(), True),
    StructField("ath", DoubleType(), True),
    StructField("ath_change_percentage", DoubleType(), True),
    StructField("ath_date", StringType(), True),
    StructField("atl", DoubleType(), True),
    StructField("atl_change_percentage", DoubleType(), True),
    StructField("atl_date", StringType(), True),
    StructField("roi", StringType(), True),
    StructField("last_updated", StringType(), True)
])

raw_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVER) \
    .option("subscribe", EVENTHUB_NAME) \
    .option("kafka.security.protocol", "SASL_SSL") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option("kafka.sasl.jaas.config", SASL_JAAS) \
    .option("startingOffsets", "earliest") \
    .load()

In [0]:
# ── Step 2: Decode bytes → JSON string → parsed struct ────────────────────────
# raw_df.value is BinaryType; cast to StringType gives you the raw JSON string
parsed_df = (
    raw_df
        .withColumn("json_str",  col("value").cast("string"))          # bytes → JSON string
        .withColumn("data",      from_json(col("json_str"), schema))   # JSON string → struct
        .withColumn("kafka_key", col("key").cast("string"))            # key bytes → string (optional)
        .withColumn("ingest_ts", current_timestamp())                  # audit: when Spark processed it
)

# ── Step 3: Flatten the struct into top-level columns ─────────────────────────
bronze_df = parsed_df.select(
    # Kafka metadata (useful for debugging / deduplication)
    col("kafka_key"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp").alias("event_hub_enqueue_ts"),
    col("ingest_ts"),

    # All crypto fields unpacked from the struct
    col("data.id"),
    col("data.symbol"),
    col("data.name"),
    col("data.current_price"),
    col("data.market_cap"),
    col("data.market_cap_rank"),
    col("data.fully_diluted_valuation"),
    col("data.total_volume"),
    col("data.high_24h"),
    col("data.low_24h"),
    col("data.price_change_24h"),
    col("data.price_change_percentage_24h"),
    col("data.market_cap_change_24h"),
    col("data.market_cap_change_percentage_24h"),
    col("data.circulating_supply"),
    col("data.total_supply"),
    col("data.max_supply"),
    col("data.ath"),
    col("data.ath_change_percentage"),
    col("data.ath_date"),
    col("data.atl"),
    col("data.atl_change_percentage"),
    col("data.atl_date"),
    col("data.roi"),
    col("data.last_updated"),
)

# ── Step 4: Write to ADLS Bronze layer (Delta format recommended) ─────────────
# query = (
#     bronze_df.writeStream
#         .format("delta")                          # use parquet if Delta not available
#         .outputMode("append")
#         .option("checkpointLocation", CHECKPOINT)
#         .option("mergeSchema", "true")            # handles schema evolution gracefully
#         .trigger(availableNow = True)
#         .start(OUTPUT)
# )

# query.awaitTermination()

In [0]:
query = (
    bronze_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"{CHECKPOINT}/bronze_checkpoints_3/")
        .trigger(processingTime="1 minute")  
        .start(OUTPUT)
)